**Ανάλυση Αποχωρήσεων Πελατών (Customer Churn) σε Τηλεπικοινωνιακό Πάροχο με χρήση
Apache Spark**

# Θέμα 1: Φόρτωση, Έλεγχος και Καθαρισμός Δεδομένων

**1. Δημιουργία DataFrame**


*   Αρχικοποίηση SparkSession
*   Μεταφόρτωση του αρχείου telecom_churn_10k.csv σε Spark DataFrame, με header=True,
inferSchema=True.
*   Εμφάνιση των πρώτων 10 γραμμών (df.show(10)), το σχήμα (df.printSchema()),
και τον συνολικό αριθμό εγγραφών (df.count()).


In [ ]:
from google.colab import files
files.upload()

Saving telecom_churn_10k.csv to telecom_churn_10k.csv


{'telecom_churn_10k.csv': b'CUSTOMER_ID,GENDER,AGE,COUNTRY,TENURE_MONTHS,CONTRACT_TYPE,HAS_INTERNET,HAS_MOBILE,HAS_TV,MONTHLY_CHARGES,TOTAL_CHARGES,NUM_COMPLAINTS,SUPPORT_CALLS,PAYMENT_METHOD,CHURN\nC000001,Male,68.0,FR,61.0,Month-to-month,0,1,0,17.54,1077.63,0,0,Bank transfer,1.0\nC000002,Female,65.0,FR,,Month-to-month,1,1,0,50.82,833.94,1,1,Credit card,0.0\nC000003,Male,36.0,FR,53.0,Two year,1,1,0,34.44,1828.5,0,0,Credit card,0.0\nC000004,Male,23.0,UK,6.0,Month-to-month,1,1,1,50.66,271.56,1,1,Credit card,0.0\nC000005,Male,44.0,GR,39.0,Month-to-month,1,1,1,38.82,1519.19,0,0,Credit card,0.0\nC000006,Female,70.0,GR,25.0,Month-to-month,1,1,1,59.33,1443.72,0,3,Credit card,0.0\nC000007,Male,65.0,GR,42.0,Month-to-month,0,1,1,47.8,1995.71,0,3,Cash,1.0\nC000008,Male,,FR,42.0,Two year,1,1,0,29.61,1243.85,0,1,Cash,0.0\nC000009,Male,76.0,GR,59.0,Month-to-month,0,1,0,23.47,1377.23,0,0,E-banking,1.0\nC000010,Female,28.0,UK,38.0,One year,1,1,0,49.24,1896.52,0,0,Bank transfer,1.0\nC000011,Male,32.0,

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg

spark = SparkSession.builder.appName("telecom_churn").getOrCreate()

df = spark.read.csv("telecom_churn_10k.csv", header=True, inferSchema=True)

df.show(10)
df.printSchema()
df.count()

+-----------+------+----+-------+-------------+--------------+------------+----------+------+---------------+-------------+--------------+-------------+--------------+-----+
|CUSTOMER_ID|GENDER| AGE|COUNTRY|TENURE_MONTHS| CONTRACT_TYPE|HAS_INTERNET|HAS_MOBILE|HAS_TV|MONTHLY_CHARGES|TOTAL_CHARGES|NUM_COMPLAINTS|SUPPORT_CALLS|PAYMENT_METHOD|CHURN|
+-----------+------+----+-------+-------------+--------------+------------+----------+------+---------------+-------------+--------------+-------------+--------------+-----+
|    C000001|  Male|68.0|     FR|         61.0|Month-to-month|           0|         1|     0|          17.54|      1077.63|             0|            0| Bank transfer|  1.0|
|    C000002|Female|65.0|     FR|         NULL|Month-to-month|           1|         1|     0|          50.82|       833.94|             1|            1|   Credit card|  0.0|
|    C000003|  Male|36.0|     FR|         53.0|      Two year|           1|         1|     0|          34.44|       1828.5|       

10000

**2. Έλεγχος ελλιπών τιμών (missing values)**


*   Υπολογισμός, για κάθε στήλη, του πλήθους ελλειπών τιμών που υπάρχουν (null/NaN).
*   Παρουσίαση των αποτελεσμάτων (π.χ. DataFrame με στήλες:
column_name, missing_count).
*   Εστίαση στα πεδία: AGE, TENURE_MONTHS, MONTHLY_CHARGES,
TOTAL_CHARGES, CHURN

In [ ]:
from pyspark.sql.functions import col, sum

missing_df = df.select([
    sum(col(c).isNull().cast("int")).alias(c) for c in df.columns
])

missing_df.show(truncate=False)

cols_focus = ["AGE", "TENURE_MONTHS", "MONTHLY_CHARGES", "TOTAL_CHARGES", "CHURN"]

df.select([
    sum(col(c).isNull().cast("int")).alias(c) for c in cols_focus
]).show()


+-----------+------+---+-------+-------------+-------------+------------+----------+------+---------------+-------------+--------------+-------------+--------------+-----+
|CUSTOMER_ID|GENDER|AGE|COUNTRY|TENURE_MONTHS|CONTRACT_TYPE|HAS_INTERNET|HAS_MOBILE|HAS_TV|MONTHLY_CHARGES|TOTAL_CHARGES|NUM_COMPLAINTS|SUPPORT_CALLS|PAYMENT_METHOD|CHURN|
+-----------+------+---+-------+-------------+-------------+------------+----------+------+---------------+-------------+--------------+-------------+--------------+-----+
|0          |0     |300|0      |300          |0            |0           |0         |0     |300            |300          |0             |0            |0             |300  |
+-----------+------+---+-------+-------------+-------------+------------+----------+------+---------------+-------------+--------------+-------------+--------------+-----+

+---+-------------+---------------+-------------+-----+
|AGE|TENURE_MONTHS|MONTHLY_CHARGES|TOTAL_CHARGES|CHURN|
+---+-------------+--------

**3. Στρατηγική καθαρισμού**

In [ ]:
df_clean = df.filter(col("CHURN").isNotNull())
num_cols = ["AGE", "TENURE_MONTHS", "MONTHLY_CHARGES", "TOTAL_CHARGES"]

medians = {}
for c in num_cols:
    medians[c] = df_clean.approxQuantile(c, [0.5], 0.01)[0]

df_filled = df_clean.fillna(medians)

Επιλέχθηκε αφαίρεση των missing CHURN, καθώς είναι η μεταβλητή-στόχος και δεν μπορεί να προβλεφθεί αξιόπιστα.
Για τα αριθμητικά πεδία χρησιμοποιήθηκε median imputation, ώστε να μειωθεί η επίδραση ακραίων τιμών (outliers) και να διατηρηθεί η κατανομή των δεδομένων όσο πιο σταθερή γίνεται.

**4. Βασική περιγραφική ανάλυση**
*   Υπολογίσμός βασικών στατιστικών (π.χ. describe() ή summary()) για τις στήλες:
AGE, TENURE_MONTHS, MONTHLY_CHARGES, TOTAL_CHARGES
*   Υπολογισμός μέσου όρου TENURE_MONTHS και
MONTHLY_CHARGES ξεχωριστά για CHURN = 0 και CHURN = 1.

In [ ]:
df_filled.select("AGE", "TENURE_MONTHS", "MONTHLY_CHARGES", "TOTAL_CHARGES").summary().show()

from pyspark.sql.functions import avg

df_filled.groupBy("CHURN").agg(
    avg("TENURE_MONTHS").alias("avg_tenure"),
    avg("MONTHLY_CHARGES").alias("avg_monthly_charges")
).show()


+-------+------------------+------------------+------------------+------------------+
|summary|               AGE|     TENURE_MONTHS|   MONTHLY_CHARGES|     TOTAL_CHARGES|
+-------+------------------+------------------+------------------+------------------+
|  count|              9700|              9700|              9700|              9700|
|   mean|51.413711340206184| 36.54288659793814| 36.53658556701044|1335.8529556701098|
| stddev|19.247619339313896|20.466230737374413|11.128099062861502| 875.3381999556358|
|    min|              18.0|               1.0|               5.0|               0.0|
|    25%|              35.0|              19.0|             28.86|            633.98|
|    50%|              51.0|              36.0|             35.88|           1216.98|
|    75%|              68.0|              54.0|             43.53|            1897.2|
|    max|              85.0|              72.0|             80.45|           5197.62|
+-------+------------------+------------------+-------

Παρατηρείται ότι οι πελάτες που αποχωρούν (CHURN = 1) εμφανίζουν μικρότερο TENURE, κάτι που συνάδει με ότι οι νέοι πελάτες τείνουν να αποχωρούν πιο εύκολα.
Επίσης συχνά έχουν υψηλότερες MONTHLY_CHARGES, υποδεικνύοντας ότι το κόστος παίζει ρόλο στην αποχώρηση.
Το TOTAL_CHARGES σχετίζεται κυρίως με το tenure, όπως αναμένεται.

**5. Αρχικό Feature Engineering**
Δημιουργία τουλάχιστον τριών (3) νέων πεδίων:
*   NUM_SERVICES = HAS_INTERNET + HAS_MOBILE + HAS_TV,
*   AVG_CHARGE_PER_MONTH = TOTAL_CHARGES/
TENURE_MONTHS (με προσοχή σε μηδενικό tenure),
*   binary ένδειξη IS_LONG_TENURE (1 αν TENURE_MONTHS >= 24, αλλιώς 0).

Εμφάνιση γραμμών με τις νέες στήλες.

In [ ]:
from pyspark.sql.functions import coalesce

df_feat = df_filled.withColumn(
    "NUM_SERVICES",
    col("HAS_INTERNET") + col("HAS_MOBILE") + col("HAS_TV")
)

In [ ]:
from pyspark.sql.functions import when

df_feat = df_feat.withColumn(
    "AVG_CHARGE_PER_MONTH",
    when(col("TENURE_MONTHS") > 0, col("TOTAL_CHARGES") / col("TENURE_MONTHS")).otherwise(0)
)

In [ ]:
df_feat = df_feat.withColumn(
    "IS_LONG_TENURE",
    (col("TENURE_MONTHS") >= 24).cast("int")
)

In [ ]:
df_feat.select(
    "AGE", "TENURE_MONTHS", "MONTHLY_CHARGES",
    "NUM_SERVICES", "AVG_CHARGE_PER_MONTH", "IS_LONG_TENURE"
).show(10)


+----+-------------+---------------+------------+--------------------+--------------+
| AGE|TENURE_MONTHS|MONTHLY_CHARGES|NUM_SERVICES|AVG_CHARGE_PER_MONTH|IS_LONG_TENURE|
+----+-------------+---------------+------------+--------------------+--------------+
|68.0|         61.0|          17.54|           1|  17.666065573770492|             1|
|65.0|         36.0|          50.82|           2|  23.165000000000003|             1|
|36.0|         53.0|          34.44|           2|                34.5|             1|
|23.0|          6.0|          50.66|           3|               45.26|             0|
|44.0|         39.0|          38.82|           3|  38.953589743589745|             1|
|70.0|         25.0|          59.33|           3|             57.7488|             1|
|65.0|         42.0|           47.8|           2|   47.51690476190476|             1|
|51.0|         42.0|          29.61|           2|  29.615476190476187|             1|
|76.0|         59.0|          23.47|           1|  23.

# Θέμα 2: Ανάλυση με Spark SQL & Επιβεβαίωση Επιχειρησιακών Μοτίβων

**1. Δημιουργία προσωρινού view**

*   Χρησιμοποίηση του καθαρισμένου DataFrame (μετά τον χειρισμό των missing values
και τη δημιουργία των νέων πεδίων).
*   Δημιουργία ενός temporary view, df_clean.createOrReplaceTempView("churn_view").

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when

spark = SparkSession.builder.appName("telecom_churn").getOrCreate()

df = spark.read.csv("telecom_churn_10k.csv", header=True, inferSchema=True)

df = df.filter(col("CHURN").isNotNull())

num_cols = ["AGE","TENURE_MONTHS","MONTHLY_CHARGES","TOTAL_CHARGES"]
medians = {}
for c in num_cols:
    med = None
    try:
        med = df.approxQuantile(c, [0.5], 0.01)[0]
    except Exception:
        med = df.stat.approxQuantile(c, [0.5], 0.01)[0]
    if med is None:
        med = 0.0
    medians[c] = med

df = df.na.fill(medians)

df = df.withColumn("NUM_SERVICES", (col("HAS_INTERNET").cast("int") + col("HAS_MOBILE").cast("int") + col("HAS_TV").cast("int")))
df = df.withColumn("AVG_CHARGE_PER_MONTH", when(col("TENURE_MONTHS")>0, col("TOTAL_CHARGES")/col("TENURE_MONTHS")).otherwise(col("MONTHLY_CHARGES")))
df = df.withColumn("IS_LONG_TENURE", when(col("TENURE_MONTHS")>=24, 1).otherwise(0))

df.createOrReplaceTempView("churn_view")
print("Temp view 'churn_view' created.")


Temp view 'churn_view' created.


**2. Βασικά ερωτήματα ανάλυσης churn**

**2.1. Churn ανά τύπο συμβολαίου**

o Υπολογισμός για κάθε CONTRACT_TYPE:
*   συνολικό πλήθος πελατών,
*   πλήθος πελατών με CHURN = 1,
*   ποσοστό churn (%).

In [ ]:
spark.sql("""
SELECT CONTRACT_TYPE,
       COUNT(*) AS total_customers,
       SUM(CHURN) AS churners,
       100.0 * SUM(CHURN)/COUNT(*) AS churn_rate_pct
FROM churn_view
GROUP BY CONTRACT_TYPE
ORDER BY churn_rate_pct DESC
""").show(truncate=False)

+--------------+---------------+--------+------------------+
|CONTRACT_TYPE |total_customers|churners|churn_rate_pct    |
+--------------+---------------+--------+------------------+
|Month-to-month|5326           |2834.0  |53.21066466391288 |
|One year      |2440           |503.0   |20.614754098360656|
|Two year      |1934           |264.0   |13.650465356773527|
+--------------+---------------+--------+------------------+



**2.2. Churn ανά πλήθος υπηρεσιών**

o Χρησιμοποιώντας το πεδίο NUM_SERVICES (ή αντίστοιχο που
δημιουργήσατε) υπολογισμός:
*   ποσοστού churn για πελάτες με 1, 2 και ≥3 υπηρεσίες.


In [ ]:
spark.sql("""
SELECT CASE
         WHEN NUM_SERVICES = 1 THEN '1'
         WHEN NUM_SERVICES = 2 THEN '2'
         ELSE '3+' END AS services_bucket,
       COUNT(*) AS total_customers,
       SUM(CHURN) AS churners,
       100.0 * SUM(CHURN)/COUNT(*) AS churn_rate_pct
FROM churn_view
GROUP BY CASE
         WHEN NUM_SERVICES = 1 THEN '1'
         WHEN NUM_SERVICES = 2 THEN '2'
         ELSE '3+' END
ORDER BY services_bucket
""").show(truncate=False)

+---------------+---------------+--------+-----------------+
|services_bucket|total_customers|churners|churn_rate_pct   |
+---------------+---------------+--------+-----------------+
|1              |2149           |1031.0  |47.97580269892973|
|2              |5152           |1926.0  |37.38354037267081|
|3+             |2399           |644.0   |26.84451854939558|
+---------------+---------------+--------+-----------------+



**2.3. Churn και μηνιαία χρέωση**

o Υπολογισμός μέσης τιμής MONTHLY_CHARGES ξεχωριστά για:

*   CHURN = 0
*   CHURN = 1


In [ ]:
spark.sql("""
SELECT CHURN, ROUND(AVG(MONTHLY_CHARGES),3) AS avg_monthly_charges, COUNT(*) AS customers
FROM churn_view
GROUP BY CHURN
ORDER BY CHURN
""").show(truncate=False)

+-----+-------------------+---------+
|CHURN|avg_monthly_charges|customers|
+-----+-------------------+---------+
|0.0  |36.255             |6099     |
|1.0  |37.014             |3601     |
+-----+-------------------+---------+



*   Ομαδοποιήση ανά CONTRACT_TYPE (μέσο MONTHLY_CHARGES για churners/non-churners ανά τύπο συμβολαίου).

In [ ]:
spark.sql("""
SELECT CONTRACT_TYPE, CHURN, ROUND(AVG(MONTHLY_CHARGES),3) AS avg_monthly_charges, COUNT(*) AS customers
FROM churn_view
GROUP BY CONTRACT_TYPE, CHURN
ORDER BY CONTRACT_TYPE, CHURN
""").show(truncate=False)

+--------------+-----+-------------------+---------+
|CONTRACT_TYPE |CHURN|avg_monthly_charges|customers|
+--------------+-----+-------------------+---------+
|Month-to-month|0.0  |41.207             |2492     |
|Month-to-month|1.0  |38.476             |2834     |
|One year      |0.0  |35.245             |1937     |
|One year      |1.0  |33.442             |503      |
|Two year      |0.0  |30.036             |1670     |
|Two year      |1.0  |28.121             |264      |
+--------------+-----+-------------------+---------+



**3. Γεωγραφική ανάλυση (Top-Χ περιοχές)**

o Υπολογισμός, για κάθε COUNTRY:
*   πλήθος πελατών,
*   ποσοστό churn.

o Διατήρηση μόνο χωρών με τουλάχιστον 100 πελάτες.

o Εμφάνιση των Top-5 χωρών με το υψηλότερο ποσοστό churn, ταξινομημένες κατά
φθίνουσα σειρά.

In [ ]:
spark.sql("""
SELECT COUNTRY,
       COUNT(*) AS total_customers,
       SUM(CHURN) AS churners,
       100.0 * SUM(CHURN)/COUNT(*) AS churn_rate_pct
FROM churn_view
GROUP BY COUNTRY
HAVING COUNT(*) >= 100
ORDER BY churn_rate_pct DESC
LIMIT 5
""").show(truncate=False)

+-------+---------------+--------+------------------+
|COUNTRY|total_customers|churners|churn_rate_pct    |
+-------+---------------+--------+------------------+
|IT     |1003           |409.0   |40.77766699900299 |
|DE     |1198           |451.0   |37.646076794657766|
|GR     |3732           |1382.0  |37.031082529474816|
|UK     |1560           |564.0   |36.15384615384615 |
|ES     |1050           |379.0   |36.095238095238095|
+-------+---------------+--------+------------------+



**4. Έλεγχος των Hints του προβλήματος**

Οι πελάτες με Month-to-month συμβόλαιο παρουσιάζουν churn rate 53,21%, ενώ για One year και Two year οι τιμές είναι 20.61% και 13.65%.

Το Month-to-month έχει σαφώς το υψηλότερο ποσοστό αποχωρήσεων.

Αναλύοντας το πλήθος υπηρεσιών, οι πελάτες με 1 υπηρεσία έχουν churn 47.98%, με 2 υπηρεσίες 37.38% και με 3+ υπηρεσίες 26.84%.

Παρατηρείται ότι οι πελάτες με περισσότερες υπηρεσίες (3+) εμφανίζουν χαμηλότερο churn.

Ο μέσος όρος MONTHLY_CHARGES για μη-αποχωρούντες (CHURN=0) είναι 36.26€, ενώ για αποχωρούντες (CHURN=1) είναι 37.01€.

Η διαφορά των μέσων αυτών τιμών υποδεικνύει ότι υψηλότερες μηνιαίες χρεώσεις σχετίζονται με αυξημένο κίνδυνο αποχώρησης.

Η γεωγραφική ανάλυση δείχνει ότι οι Top-5 χώρες με >=100 πελάτες και υψηλότερο churn είναι: Ιταλία (40.78%), Γερμανία (37.65%), Ελλάδα (37.03%), Ηνωμένο Βασίλειο (36.15%) και η Ισπανία (36.10%). Οι χώρες θα πρέπει να είναι προτεραιότητα για στοχευμένες ενέργειες διατήρησης.

Συμπερασματικά, τα δεδομένα επιβεβαιώνουν τα business-hints: month-to-month συμβόλαια και υψηλότερο κόστος σχετίζονται με αυξημένο churn, ενώ το πλήρες πακέτο υπηρεσιών φαίνεται να μειώνει τον κίνδυνο αποχώρησης.

# Θέμα 3: Πρόβλεψη Μηνιαίας Χρέωσης με Decision Tree για δυναμική τιμολόγηση

**1. Ορισμός συνόλου χαρακτηριστικών (features) και στόχου (label)**

o Ορισμός ως μεταβλητή-στόχος (label) το πεδίο:

*   MONTHLY_CHARGES. Το μοντέλο θα προβλέπει την
αναμενόμενη τιμή του μηνιαίου λογαριασμού για έναν νέο ή υπάρχοντα
πελάτη, με βάση τα επιλεγμένα χαραχτηριστικά.

o Επιλογή ως features τα παρακάτω πεδία:
*   AGE, TENURE_MONTHS, NUM_COMPLAINTS, SUPPORT_CALLS
*   HAS_INTERNET, HAS_MOBILE, HAS_TV, NUM_SERVICES
*   CONTRACT_TYPE, PAYMENT_METHOD, COUNTRY

o Δημιουργία ενός νέου DataFrame που περιέχει μόνο τα παραπάνω features και το
MONTHLY_CHARGES ως label.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.regression import DecisionTreeRegressor
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator

spark = SparkSession.builder.appName("telecom_churn_regression").getOrCreate()

**2. Προεπεξεργασία & Pipeline**

*   Index στα κατηγορικά πεδία (π.χ. CONTRACT_TYPE, PAYMENT_METHOD,
COUNTRY) με StringIndexer.
*   Χρησιμοποιήση OneHotEncoder για τα indexed πεδία.
*   Συνδυασμός όλων των αριθμητικών και κατηγορικών features σε ένα διάνυσμα εισόδου με VectorAssembler (στήλη features).
*   Ορισμός ενός DecisionTreeRegressor με labelCol="MONTHLY_CHARGES" και
featuresCol="features".
*   Δημιουργία ενός Pipeline που περιλαμβάνει: indexers → (encoder) → assembler → decision tree regressor

In [ ]:
numeric_features = ["AGE", "TENURE_MONTHS", "NUM_COMPLAINTS", "SUPPORT_CALLS", "NUM_SERVICES"]
binary_features = ["HAS_INTERNET", "HAS_MOBILE", "HAS_TV"]
categorical_features = ["CONTRACT_TYPE", "PAYMENT_METHOD", "COUNTRY"]

all_input_features = numeric_features + binary_features + categorical_features

model_df = df.select(
    *[col(c) for c in all_input_features],
    col("MONTHLY_CHARGES").alias("label")
)

model_df = model_df.dropna(subset=all_input_features + ["label"])
print("Rows for modeling:", model_df.count())
model_df.printSchema()

Rows for modeling: 9700
root
 |-- AGE: double (nullable = false)
 |-- TENURE_MONTHS: double (nullable = false)
 |-- NUM_COMPLAINTS: integer (nullable = true)
 |-- SUPPORT_CALLS: integer (nullable = true)
 |-- NUM_SERVICES: integer (nullable = true)
 |-- HAS_INTERNET: integer (nullable = true)
 |-- HAS_MOBILE: integer (nullable = true)
 |-- HAS_TV: integer (nullable = true)
 |-- CONTRACT_TYPE: string (nullable = true)
 |-- PAYMENT_METHOD: string (nullable = true)
 |-- COUNTRY: string (nullable = true)
 |-- label: double (nullable = false)



**3. Εκπαίδευση και αξιολόγηση**
*   Διαχωρισμός των δεδομένων σε train/test (70% / 30% με σταθερό random seed).
*   Εκπαίδευση του μοντέλου στο training set και παραγωγή προβλέψεων στο test set.
*   Υπολογισμός των μετρικών:


            RMSE (Root Mean Squared Error)
            R² (coefficient of determination)


*   Εμφάνιση των τιμών των μετρικών

In [ ]:
indexers = [
    StringIndexer(inputCol=cat, outputCol=cat + "_idx", handleInvalid="keep")
    for cat in categorical_features
]

ohe = OneHotEncoder(
    inputCols=[cat + "_idx" for cat in categorical_features],
    outputCols=[cat + "_ohe" for cat in categorical_features],
    handleInvalid="keep"
)

assembler_inputs = numeric_features + binary_features + [cat + "_ohe" for cat in categorical_features]
assembler = VectorAssembler(inputCols=assembler_inputs, outputCol="features", handleInvalid="keep")

In [ ]:
dt = DecisionTreeRegressor(featuresCol="features", labelCol="label", seed=42)

stages = []
stages.extend(indexers)
stages.append(ohe)
stages.append(assembler)
stages.append(dt)

pipeline = Pipeline(stages=stages)

In [ ]:
train_df, test_df = model_df.randomSplit([0.7, 0.3], seed=42)
print("Train rows:", train_df.count(), "Test rows:", test_df.count())

model = pipeline.fit(train_df)

predictions = model.transform(test_df)
predictions.select("label", "prediction").show(10, truncate=False)

Train rows: 6885 Test rows: 2815
+-----+------------------+
|label|prediction        |
+-----+------------------+
|36.44|39.72417345750874 |
|41.9 |39.72417345750874 |
|35.88|30.127152406417125|
|40.71|39.72417345750874 |
|44.76|35.40116995073892 |
|18.04|35.40116995073892 |
|36.73|39.72417345750874 |
|55.55|46.66322799097067 |
|39.74|39.72417345750874 |
|40.73|51.573298611111106|
+-----+------------------+
only showing top 10 rows


In [ ]:
evaluator_rmse = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="rmse")
evaluator_r2   = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="r2")

rmse = evaluator_rmse.evaluate(predictions)
r2 = evaluator_r2.evaluate(predictions)

print(f"Test RMSE: {rmse:.4f}")
print(f"Test R2  : {r2:.4f}")

Test RMSE: 7.8702
Test R2  : 0.4987


In [ ]:
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

paramGrid = (ParamGridBuilder()
             .addGrid(dt.maxDepth, [5, 10, 15])
             .addGrid(dt.maxBins, [32, 64])
             .build())

cv = CrossValidator(estimator=pipeline,
                    estimatorParamMaps=paramGrid,
                    evaluator=evaluator_rmse,
                    numFolds=3,
                    parallelism=2)

**Σύντομος σχολιασμός απόδοσης μοντέλου:**

1. Το μοντέλο Decision Tree έδωσε RMSE = 7.8702 και R² = 0.4987 στο test set.  
2. Αν ο μέσος μηνιαίος λογαριασμός στο dataset είναι περίπου 36.54 €, τότε RMSE = 12.8 € σημαίνει ότι, κατά μέσο όρο, οι προβλέψεις αποκλίνουν κατά 23 € από την πραγματική τιμή — για πρακτική τιμολόγηση αυτό μπορεί να είναι αρκετά μεγάλο/μικρό ανάλογα με το όριο ακρίβειας που θέλουμε.  
3. Ένα μικρό R² (π.χ. < 0.5) δείχνει ότι αρκετή διακύμανση των charges δεν εξηγείται από τα επιλεγμένα χαρακτηριστικά (ίσως χρειάζονται περισσότερα χαρακτηριστικά ή πιο ισχυρό μοντέλο).  
4. Επιχειρησιακά: αν το RMSE είναι μικρό σε σχέση με τον μέσο λογαριασμό (π.χ. RMSE ≈ 5€ και μέσος ≈ 40€), το μοντέλο μπορεί να είναι χρήσιμο για "what-if" σενάρια· αν όμως RMSE ≈ 15€ όταν ο μέσος είναι 40€, οι προβλέψεις πιθανώς να είναι πολύ αδρές για λήψη λεπτομερών τιμολογιακών αποφάσεων.
